# Project Structure & CLI

*Notebook 28*

Notebooks are for learning and prototyping. When you're ready to turn an agent into something real (a script, a CLI tool, an API) a few structural changes make the difference between a demo and a runnable application.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

print("✅ Ready!")

---

## 🎯 The Problem

Notebooks hide a lot of setup for you.

In a real script, you need to handle structure, async execution, and secrets explicitly.

---

## 📋 Part 1: What Changes When You Leave Jupyter

Three things that work effortlessly in notebooks become explicit concerns in real code:

**Top-level `await`** works in notebooks. Scripts need `asyncio.run()`.

**Global state** is shared across notebook cells. Projects need explicit imports.

**Configuration**: the notebook setup cell loads `.env` for you. Scripts must load it at startup, before anything uses the key.

---

## 🏗️ Part 2: A Reusable Project Structure

This structure works for most agent projects, from a simple CLI tool to a small web service:

```
my_agent_project/
├── .env                    # API key (never commit this)
├── .gitignore              # include .env
├── requirements.txt        # openai-agents, python-dotenv, ...
│
├── config.py                # constants: MODEL, REASONING_MODEL, paths
├── tools.py                # all @function_tool definitions
├── agent.py                # Agent and Runner: creates and runs the agent
└── main.py                 # entry point: CLI, loop, or API handler
```

For larger projects, split `tools.py` into a `tools/` folder with one file per domain.

For a web service, `main.py` becomes a FastAPI or Flask app.

---

## 📄 Part 3: The Four Files

These are printed templates you can copy directly into real project files. They are not executed in this notebook.

This matters because once your agent lives in files instead of notebook cells, you can run it from the terminal, reuse it across entry points (CLI, streaming, API), and usually change one part without touching the rest.

In [ ]:
# config.py: constants and environment loading
# -----------------------------------------------
CONFIG_TEMPLATE = '''
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path(__file__).parent / ".env")

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

# os.getenv() reads from environment: works for .env locally and platform secrets in production
# Never hardcode keys in this file. Add .env to .gitignore immediately.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not set. Check your .env or platform secrets")

# Add project-specific constants here
# MAX_RETRIES = 3
# DB_PATH = Path(__file__).parent / "data" / "agent.sqlite"
'''

print("config.py:")
print(CONFIG_TEMPLATE)

#### tools.py

In [ ]:
# tools.py: all @function_tool definitions in one place
# -------------------------------------------------------
TOOLS_TEMPLATE = '''
from agents import function_tool


@function_tool
def my_tool(query: str) -> str:
    """One-line description of what this tool does."""
    return f"Result for: {query}"


# Add more tools here, all imported into agent.py
# For real tools, handle expected failures with the Lesson 06 patterns
'''

print("tools.py:")
print(TOOLS_TEMPLATE)

#### agent.py

In [ ]:
# agent.py: agent definition and async run functions
# ---------------------------------------------------
AGENT_TEMPLATE = '''
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from config import MODEL
from tools import my_tool


instructions = "Your agent instructions here."

agent = Agent(
    name="MyAgent",
    instructions=instructions,
    model=MODEL,
    tools=[my_tool]
)


async def run_agent(message: str) -> str:
    """Run one independent turn (no session memory: Lesson 17 adds it)."""

    result = await Runner.run(agent, input=message)

    return result.final_output


async def run_agent_streamed(message: str):
    """Run one independent turn, yielding the response as text chunks."""

    result = Runner.run_streamed(agent, input=message)

    async for event in result.stream_events():
        if isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                yield data.delta
'''

print("agent.py:")
print(AGENT_TEMPLATE)

#### main.py

In [ ]:
# main.py: CLI entry point
# --------------------------
MAIN_TEMPLATE = '''
import asyncio
from agent import run_agent


async def main():
    print("Agent ready. Type 'quit' to exit.")
    # Each turn is independent: the loop carries no conversation history
    while True:
        message = input("\\nYou: ").strip()
        if message.lower() in ["quit", "exit", "q"]:
            break
        if not message:
            continue
        response = await run_agent(message)
        print(f"Agent: {response}")


if __name__ == "__main__":
    asyncio.run(main())
'''

print("main.py:")
print(MAIN_TEMPLATE)

### 💡 Key Takeaway

Each file has one responsibility, so adding a tool or chasing a bug usually has an obvious home.

This is the simplest CLI-style entry point. For richer argument handling (flags, subcommands), layer `argparse` or a library like `typer` on top of the same structure.

---

## 📡 Part 4: Streaming Agent Output in a Script

`Runner.run()` returns only when the run completes. Streaming shows the response as it is generated, a much better terminal experience.

`agent.py` adds `run_agent_streamed()`, built on `Runner.run_streamed()`: same agent, same SDK, and `main.py` just prints the chunks as they arrive.

In [ ]:
# main.py: streaming version
# ----------------------------
STREAMING_TEMPLATE = '''
import asyncio
from agent import run_agent_streamed


async def run_streaming(message: str) -> None:
    """Print the agent's response as it streams."""
    print("Agent: ", end="", flush=True)

    async for chunk in run_agent_streamed(message):
        print(chunk, end="", flush=True)

    print()  # newline after streaming completes


async def main():
    print("Agent ready (streaming). Type 'quit' to exit.")
    while True:
        message = input("\\nYou: ").strip()
        if message.lower() in ["quit", "exit", "q"]:
            break
        if not message:
            continue
        await run_streaming(message)


if __name__ == "__main__":
    asyncio.run(main())
'''

print("main.py (streaming version):")
print(STREAMING_TEMPLATE)

### 💡 Key Takeaway

The agent definition is unchanged: same SDK, same tools.

What changes is how you consume the result: `run_agent_streamed()` iterates `Runner.run_streamed()`'s events with `async for`, keeping only the text deltas.

---

## 🎯 Part 5: When a Single-Purpose Agent Is Enough

Not every agent needs tools.

Use tools and handoffs when the model must choose actions or delegation.

Add sessions for conversation state and guardrails for input or output checks.

When you already know the exact call (single-turn classification, extraction, transformation), a focused no-tool agent is enough.

With no tools, handoffs, or extra checks involved, it generally completes with one model request.

For OpenAI models, the Agents SDK uses the Responses API by default under the hood. This course continues using `Agent` and `Runner`.

In [ ]:
# classifier.py: a second, single-purpose agent in its own module
# (small projects can add it to agent.py instead)
# ---------------------------------------------------------------
CLASSIFIER_TEMPLATE = '''
from typing import Literal

from pydantic import BaseModel
from agents import Agent, Runner
from config import MODEL


class Sentiment(BaseModel):
    label: Literal["positive", "negative", "neutral"]


classifier = Agent(
    name="Classifier",
    instructions="Classify the sentiment of the supplied text.",
    model=MODEL,
    output_type=Sentiment,
)


async def classify(text: str) -> Sentiment:
    """One fixed call: no tools, structured result (Lesson 05)."""
    result = await Runner.run(classifier, input=text)
    return result.final_output
'''

print("classifier.py:")
print(CLASSIFIER_TEMPLATE)

### 💡 When to Use Each

Use tools and handoffs when the model must choose actions or delegation, sessions for conversation state, and guardrails for boundary checks.

Use a single-purpose no-tool agent for fixed calls: classification, extraction, transformation.

On a successful run, structured output (Lesson 05) gives downstream code the declared shape. Refusals or validation failures raise instead.

---

## 🌐 Part 6: When to Consider a Simple API Wrapper

This is an entry-point variation, not a deployment lesson. Lesson 30 covers deploying to a live environment.

A CLI loop works for personal tools. Reach for an API wrapper when another process (a web frontend, a Slack bot, a scheduled job) needs to call your agent.

Add a thin API layer:

In [ ]:
# main.py: FastAPI version (requires: pip install fastapi uvicorn)
FASTAPI_TEMPLATE = '''
from fastapi import FastAPI
from pydantic import BaseModel
from agent import run_agent

app = FastAPI()


class ChatRequest(BaseModel):
    message: str


class ChatResponse(BaseModel):
    response: str


# Stateless: each request is one independent turn. Per-conversation memory
# needs a session keyed by a conversation id (Lesson 17).
@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    response = await run_agent(request.message)
    return ChatResponse(response=response)


# Run with: uvicorn main:app --reload
# Call with: POST http://localhost:8000/chat
#              Body: {"message": "your question here"}
'''

print("main.py (FastAPI version):")
print(FASTAPI_TEMPLATE)

### 💡 Key Takeaway

`run_agent()` in `agent.py` is already `async`. FastAPI handles the event loop, so you just `await` it directly in the route handler.

The agent code doesn't change at all.

---

## 💪 Practice Exercises

### Exercise 1: Build the Four-File Structure

*Covers: project structure (the four-file agent template)*

Create a working agent project using the templates from Part 3.

**Note:** For these exercises, work outside Jupyter. Open a terminal, create the `my_agent/` directory, and create the `.py` files with a text editor, copying the templates from this notebook.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Build the Four-File Structure
# --------------------------------------------------------------
# Objective: Turn the templates from Part 3 into a working project.

# TODO 1: Create a project directory called my_agent/ with:
#           - a .env containing OPENAI_API_KEY (or export the variable)
#           - a .gitignore that includes .env
#           - requirements.txt with the course-tested pins
#             (openai-agents==0.13.0, python-dotenv==1.2.2),
#             then pip install -r requirements.txt

# TODO 2: Create config.py using the CONFIG_TEMPLATE from Part 3

# TODO 3: Create tools.py with one @function_tool of your choice

# TODO 4: Create agent.py with an Agent that uses your tool
#           Use run_agent() as the async entry point

# TODO 5: Create main.py with the CLI loop from Part 3
#           Run it from the terminal: python main.py

# --- Write your code below this line ---

### Exercise 2: Add Streaming

*Covers: streaming output (modifying a project for streamed responses)*

Modify your main.py from Exercise 1 to stream output using the pattern from Part 4.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Add Streaming
# --------------------------------------------------------------
# Objective: Stream the agent's output in your CLI.

# TODO 1: Add run_agent_streamed() to your agent.py (Part 4), then copy
#           the streaming template from Part 4 into your main.py

# TODO 2: Run it from the terminal and watch the response stream in chunks

# TODO 3: (Optional) Compare the experience to the non-streaming version

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Leaving Jupyter requires three explicit decisions:**

- Replace top-level `await` with `asyncio.run()` in `main.py`

- Import agents and tools explicitly, no more global notebook state

- Load `.env` in `config.py` before anything uses the API key
<br>
<br>

**Four-file structure separates concerns cleanly:**

- `config.py` owns constants, `tools.py` owns tools, `agent.py` owns the agent, `main.py` owns the entry point

- Same structure scales from a CLI tool to a web service
<br>
<br>

**Streaming shows output sooner:**

- `Runner.run_streamed()`: same agent, output arrives in text chunks

- Filter `RawResponsesStreamEvent` → `ResponseTextDeltaEvent` to print text progressively
<br>
<br>

**Not every agent needs tools:**

- Tools and handoffs let the model choose actions, sessions carry state, guardrails check boundaries

- A single-purpose no-tool agent handles fixed calls: classify, extract, transform

- With nothing to orchestrate, it generally completes with one model request
<br>
<br>

**Secrets travel through environment variables:**

- `.env` for local dev, platform secrets for deployment: same `os.getenv()` call either way

- Never hardcode, never commit `.env`
<br>
<br>

**A thin API wrapper is just another entry point:**

- Swap `main.py` from a CLI loop to a FastAPI app to expose your agent to other systems

- The agent code in `agent.py` doesn't change, only the entry point does

---

## 📍 Next Step

**Notebook 29: Architecture Decisions**  

A decision framework for model selection, single vs multi-agent, memory, MCP, and when to add complexity.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-28-project-structure-and-cli)

---